# SecureReview — GRPO Training with Unsloth + HF TRL

Train `Qwen2.5-1.5B-Instruct` to improve at security code review using Group Relative Policy Optimization (GRPO) against the live SecureReview environment.

**Environment:** https://sam25kat-securereview.hf.space  
**Task:** `dependency_review` (easiest task, highest baseline score)  
**Method:** GRPO — generate 4 rollouts per prompt, score each via live env, update model toward higher-scoring outputs  
**Hardware:** Google Colab T4 (free tier or HF compute credits)

---
**Reward signal:** The environment returns a score in (0.01, 0.99) based on F1 over planted vulnerabilities.  
A model that finds more real bugs with fewer false positives scores higher.


## Cell 1 — Install Dependencies

In [ ]:
%%capture
# Install Unsloth (handles efficient 4-bit QLoRA + GRPO)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade --no-cache-dir trl peft accelerate bitsandbytes
!pip install requests matplotlib datasets

## Cell 2 — Configuration

In [ ]:
import os

# ── Environment ──────────────────────────────────────────────────────────────
ENV_URL = os.getenv("ENV_URL", "https://sam25kat-securereview.hf.space")
TASK_ID = "dependency_review"      # easiest task, best baseline for small model

# ── Model ────────────────────────────────────────────────────────────────────
MODEL_NAME  = "unsloth/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LEN = 2048
LOAD_4BIT   = True

# ── Training ─────────────────────────────────────────────────────────────────
NUM_GENERATIONS       = 4    # GRPO rollouts per prompt
MAX_NEW_TOKENS        = 600  # max tokens per completion
TRAIN_STEPS           = 150  # ~15 min on T4
LEARNING_RATE         = 2e-5
LORA_RANK             = 16
GRAD_ACCUM_STEPS      = 4

# ── Output ───────────────────────────────────────────────────────────────────
OUTPUT_DIR  = "./securereview-grpo"
PLOTS_DIR   = "./plots"
os.makedirs(PLOTS_DIR, exist_ok=True)

print(f"ENV_URL : {ENV_URL}")
print(f"Model   : {MODEL_NAME}")
print(f"Steps   : {TRAIN_STEPS}")

## Cell 3 — Load Model with Unsloth (4-bit QLoRA)

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_NAME,
    max_seq_length= MAX_SEQ_LEN,
    dtype         = None,          # auto-detect (bfloat16 on A100, float16 on T4)
    load_in_4bit  = LOAD_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                   = LORA_RANK,
    target_modules      = ["q_proj", "k_proj", "v_proj", "o_proj",
                            "gate_proj", "up_proj", "down_proj"],
    lora_alpha          = LORA_RANK,
    lora_dropout        = 0,
    bias                = "none",
    use_gradient_checkpointing = "unsloth",
    random_state        = 42,
)

print("Model loaded. Parameters:")
model.print_trainable_parameters()

## Cell 4 — Environment Helpers

Utilities for calling the live SecureReview environment.

In [ ]:
import requests
import json
import re
import time


def env_reset(task_id: str, scenario_id: str | None = None) -> dict:
    payload = {"task_id": task_id}
    if scenario_id:
        payload["scenario_id"] = scenario_id
    r = requests.post(f"{ENV_URL}/reset", json=payload, timeout=30)
    r.raise_for_status()
    return r.json()


def env_step(action: dict) -> dict:
    r = requests.post(f"{ENV_URL}/step", json={"action": action}, timeout=30)
    r.raise_for_status()
    return r.json()


def env_get_scenario_ids(task_id: str) -> list[str]:
    """Fetch all scenario IDs for a task by resetting with no scenario and reading info."""
    r = requests.get(f"{ENV_URL}/tasks", timeout=30)
    r.raise_for_status()
    return r.json()


def parse_findings(text: str) -> list[dict]:
    """Extract a JSON array of findings from model output."""
    # Try to find a JSON array in the text
    patterns = [
        r'```(?:json)?\s*(\[.*?\])\s*```',   # fenced code block
        r'(\[\s*\{.*?\}\s*\])',               # bare array
    ]
    for pattern in patterns:
        m = re.search(pattern, text, re.DOTALL)
        if m:
            try:
                return json.loads(m.group(1))
            except json.JSONDecodeError:
                continue
    return []


def run_episode(completion: str, scenario_id: str) -> float:
    """Submit a model completion against the environment and return the reward."""
    findings = parse_findings(completion)

    try:
        # Reset to the specific scenario for reproducibility
        env_reset(TASK_ID, scenario_id)

        # Submit each finding
        valid_severities = {"critical", "high", "medium", "low", "info"}
        for f in findings:
            if not isinstance(f, dict):
                continue
            finding = {
                "file":        str(f.get("file", "requirements.txt")),
                "line":        f.get("line"),
                "rule_id":     str(f.get("rule_id", "DEP-001")),
                "severity":    f.get("severity", "medium") if f.get("severity") in valid_severities else "medium",
                "description": str(f.get("description", ""))[:400],
            }
            env_step({"action_type": "report_finding", "finding": finding})

        # End episode
        result = env_step({"action_type": "mark_complete"})
        return float(result.get("reward", 0.01))

    except Exception as e:
        print(f"  [env error] {e}")
        return 0.01  # participation floor


# Quick smoke test
print("Testing environment connection...")
r = requests.get(f"{ENV_URL}/health", timeout=15)
print(f"Health: {r.json()}")

## Cell 5 — Build Training Dataset

Each training example is a prompt asking the model to review a dependency file.  
We reset each scenario to get the actual file content, then build a prompt from it.

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = """You are a senior security engineer reviewing dependency files for vulnerabilities.

Identify ALL security issues including:
- Typosquatted packages (names that misspell popular libraries, e.g. 'reqeusts' instead of 'requests')
- Known CVE-vulnerable versions (e.g. requests<2.20.0 has CVE-2018-18074)
- Hallucinated / non-existent packages that don't exist on PyPI or npm
- Suspicious or malicious packages

Output ONLY a valid JSON array of findings. Each finding must have:
  file, line (integer or null), rule_id (e.g. DEP-001), severity (critical/high/medium/low/info), description

Example output:
[
  {"file": "requirements.txt", "line": 3, "rule_id": "DEP-001", "severity": "critical", "description": "Typosquat: 'reqeusts' misspells 'requests'"}
]

If no issues found, output: []
Output ONLY the JSON array. No explanations, no markdown prose."""


def build_prompt(obs: dict) -> str:
    """Build user prompt from a /reset observation."""
    ctx = obs["observation"]["context"]
    files = ctx["files"]

    parts = [f"Task: {ctx['task_description']}\n"]
    for f in files:
        parts.append(f"\n--- {f['filename']} ---\n{f['content']}")
    parts.append("\nList all security issues as a JSON array:")
    return "".join(parts)


def build_training_dataset() -> Dataset:
    """Build dataset by fetching all dependency scenarios from the live environment."""
    examples = []
    scenario_ids = [f"dep_{i:03d}" for i in range(1, 7)]  # dep_001 through dep_006

    for sid in scenario_ids:
        try:
            obs = env_reset(TASK_ID, sid)
            prompt = build_prompt(obs)
            examples.append({
                "prompt": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": prompt},
                ],
                "scenario_id": sid,
            })
            print(f"  Loaded scenario {sid}")
        except Exception as e:
            print(f"  Skipping {sid}: {e}")

    # Repeat dataset to fill training steps (each scenario seen multiple times)
    repeats = max(1, TRAIN_STEPS // (len(examples) * NUM_GENERATIONS) + 1)
    examples = examples * repeats
    print(f"\nDataset: {len(examples)} examples ({len(scenario_ids)} scenarios × {repeats} repeats)")
    return Dataset.from_list(examples)


dataset = build_training_dataset()
print(dataset)

## Cell 6 — Reward Function

GRPO reward function: parses model output as findings JSON, runs against the live environment, returns F1-based score.

In [ ]:
reward_log: list[dict] = []   # track (step, mean_reward) for plotting
_step_counter = [0]


def securereview_reward(completions, prompts=None, scenario_id=None, **kwargs) -> list[float]:
    """GRPO reward function — evaluated once per rollout batch."""
    rewards = []

    # scenario_id comes from the dataset column
    scenario_ids = kwargs.get("scenario_id", [TASK_ID] * len(completions))
    if isinstance(scenario_ids, str):
        scenario_ids = [scenario_ids] * len(completions)

    for i, (completion, sid) in enumerate(zip(completions, scenario_ids)):
        # completions can be strings or lists of message dicts
        text = completion if isinstance(completion, str) else completion[-1]["content"]
        reward = run_episode(text, sid)
        rewards.append(reward)

    _step_counter[0] += 1
    mean_r = sum(rewards) / len(rewards)
    reward_log.append({"step": _step_counter[0], "mean_reward": mean_r})
    print(f"  Step {_step_counter[0]:>4d} | rewards: {[round(r,3) for r in rewards]} | mean: {mean_r:.3f}")
    return rewards


print("Reward function defined.")

## Cell 7 — GRPO Trainer Configuration

In [ ]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    # Core GRPO settings
    num_generations              = NUM_GENERATIONS,
    max_completion_length        = MAX_NEW_TOKENS,
    # Batch / gradient
    per_device_train_batch_size  = 1,
    gradient_accumulation_steps  = GRAD_ACCUM_STEPS,
    # Optimizer
    learning_rate                = LEARNING_RATE,
    optim                        = "adamw_8bit",
    weight_decay                 = 0.01,
    lr_scheduler_type            = "cosine",
    warmup_ratio                 = 0.05,
    # Training length
    max_steps                    = TRAIN_STEPS,
    # Logging
    logging_steps                = 5,
    save_steps                   = 50,
    output_dir                   = OUTPUT_DIR,
    # Precision
    fp16                         = not torch.cuda.is_bf16_supported(),
    bf16                         = torch.cuda.is_bf16_supported(),
    seed                         = 42,
    report_to                    = "none",
)

trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    reward_funcs     = securereview_reward,
    args             = training_args,
    train_dataset    = dataset,
)

print("Trainer ready.")

## Cell 8 — Baseline Evaluation (Before Training)

Record untrained model scores on all 6 scenarios so we have a before/after comparison.

In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)   # enable fast inference mode

SCENARIO_IDS = [f"dep_{i:03d}" for i in range(1, 7)]


def evaluate_model(scenario_ids: list[str], label: str) -> dict[str, float]:
    """Run the model on each scenario and return {scenario_id: score}."""
    scores = {}
    for sid in scenario_ids:
        obs = env_reset(TASK_ID, sid)
        prompt_text = build_prompt(obs)
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": prompt_text},
        ]
        inputs = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
        ).to("cuda")
        with torch.no_grad():
            outputs = model.generate(
                inputs, max_new_tokens=MAX_NEW_TOKENS,
                temperature=0.1, do_sample=True, pad_token_id=tokenizer.eos_token_id
            )
        completion = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
        score = run_episode(completion, sid)
        scores[sid] = score
        print(f"  [{label}] {sid}: {score:.3f}")
        time.sleep(0.5)
    return scores


print("=== Baseline evaluation (untrained model) ===")
baseline_scores = evaluate_model(SCENARIO_IDS, "before")
print(f"Baseline mean: {sum(baseline_scores.values())/len(baseline_scores):.3f}")

## Cell 9 — Train

In [ ]:
FastLanguageModel.for_training(model)   # switch back to training mode

print(f"Starting GRPO training for {TRAIN_STEPS} steps...")
print(f"Model: {MODEL_NAME} | Task: {TASK_ID} | Rollouts/step: {NUM_GENERATIONS}")
print("-" * 60)

trainer.train()

print("-" * 60)
print("Training complete.")

## Cell 10 — Post-Training Evaluation

In [ ]:
FastLanguageModel.for_inference(model)

print("=== Post-training evaluation (trained model) ===")
trained_scores = evaluate_model(SCENARIO_IDS, "after")
print(f"Trained mean: {sum(trained_scores.values())/len(trained_scores):.3f}")

print("\n=== Improvement Summary ===")
for sid in SCENARIO_IDS:
    b = baseline_scores.get(sid, 0)
    t = trained_scores.get(sid, 0)
    delta = t - b
    arrow = "▲" if delta > 0 else ("▼" if delta < 0 else "—")
    print(f"  {sid}: {b:.3f} → {t:.3f}  {arrow} {delta:+.3f}")

## Cell 11 — Plot Reward Curve

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

plt.style.use("dark_background")

steps   = [e["step"]        for e in reward_log]
rewards = [e["mean_reward"] for e in reward_log]

# Smooth with a rolling window
window = 5
smoothed = np.convolve(rewards, np.ones(window)/window, mode="valid")
smooth_steps = steps[window-1:]

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(steps, rewards, color="#ff6b35", alpha=0.3, linewidth=1, label="Raw reward")
ax.plot(smooth_steps, smoothed, color="#ff6b35", linewidth=2.5, label=f"Smoothed (window={window})")
ax.set_xlabel("Training Step", fontsize=12)
ax.set_ylabel("Episode Reward", fontsize=12)
ax.set_title("SecureReview — GRPO Training Reward Curve", fontsize=14, fontweight="bold")
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2)
fig.tight_layout()
plt.savefig(f"{PLOTS_DIR}/reward_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {PLOTS_DIR}/reward_curve.png")

## Cell 12 — Before / After Comparison Plot

In [ ]:
scenarios = list(SCENARIO_IDS)
before_vals = [baseline_scores.get(s, 0) for s in scenarios]
after_vals  = [trained_scores.get(s, 0)  for s in scenarios]

x = np.arange(len(scenarios))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars_before = ax.bar(x - width/2, before_vals, width, label="Before training", color="#444444")
bars_after  = ax.bar(x + width/2, after_vals,  width, label="After training",  color="#ff6b35")

ax.set_xlabel("Scenario", fontsize=12)
ax.set_ylabel("Reward Score", fontsize=12)
ax.set_title("SecureReview — Before vs After GRPO Training", fontsize=14, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels([s.replace("dep_", "Dep ") for s in scenarios], rotation=15)
ax.set_ylim(0, 1)
ax.legend(fontsize=11)
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))

# Annotate improvement
for i, (b, a) in enumerate(zip(before_vals, after_vals)):
    delta = a - b
    color = "#22d3ee" if delta >= 0 else "#ef4444"
    ax.text(i + width/2, a + 0.02, f"{delta:+.2f}", ha="center", fontsize=9, color=color)

mean_before = sum(before_vals) / len(before_vals)
mean_after  = sum(after_vals)  / len(after_vals)
ax.text(0.98, 0.92, f"Mean: {mean_before:.2f} → {mean_after:.2f}  ({mean_after - mean_before:+.2f})",
        transform=ax.transAxes, ha="right", fontsize=10, color="white",
        bbox=dict(boxstyle="round", facecolor="#1a1a1a", alpha=0.8))

fig.tight_layout()
plt.savefig(f"{PLOTS_DIR}/before_after.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {PLOTS_DIR}/before_after.png")

## Cell 13 — Save Trained Model

In [ ]:
# Save LoRA adapters (lightweight, push to HF Hub)
model.save_pretrained("securereview-lora")
tokenizer.save_pretrained("securereview-lora")
print("LoRA adapters saved to ./securereview-lora")

# Optional: push to HF Hub
# from huggingface_hub import notebook_login
# notebook_login()
# model.push_to_hub("sam25kat/securereview-grpo-qwen2.5-1.5b")
# tokenizer.push_to_hub("sam25kat/securereview-grpo-qwen2.5-1.5b")

## Cell 14 — Print Final Summary

In [ ]:
mean_before = sum(baseline_scores.values()) / len(baseline_scores)
mean_after  = sum(trained_scores.values())  / len(trained_scores)

print("=" * 50)
print("  SecureReview GRPO Training — Final Results")
print("=" * 50)
print(f"  Model        : {MODEL_NAME}")
print(f"  Task         : {TASK_ID}")
print(f"  Training steps: {TRAIN_STEPS}")
print(f"  Mean before  : {mean_before:.3f}")
print(f"  Mean after   : {mean_after:.3f}")
print(f"  Improvement  : {mean_after - mean_before:+.3f}")
print("=" * 50)
print(f"  Plots saved  : {PLOTS_DIR}/reward_curve.png")
print(f"                 {PLOTS_DIR}/before_after.png")
print("=" * 50)